# AxiraNews Backend — Load Test Simulation
Async HTTP load test against the local API. Measures latency, throughput, and rate-limit behavior.

In [ ]:
import asyncio
import aiohttp
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List

plt.style.use('dark_background')
plt.rcParams.update({'axes.facecolor': '#0A0A0C', 'figure.facecolor': '#070708',
                     'text.color': '#F0EDE6', 'axes.labelcolor': '#C8A97E',
                     'xtick.color': '#4A4A56', 'ytick.color': '#4A4A56',
                     'axes.edgecolor': '#252528', 'grid.color': '#1A1A1E'})

API_BASE    = 'http://10.0.0.75:4000'
ENDPOINTS   = [
    '/v1/news',
    '/v1/news/top-headlines',
    '/health',
]

@dataclass
class Result:
    endpoint: str
    status:   int
    latency:  float  # ms
    ts:       float  # epoch

print('Load test config ready ✓')
print(f'Target: {API_BASE}')

In [ ]:
# ── Core async load runner ────────────────────────────────────────────────────
async def hit(session: aiohttp.ClientSession, endpoint: str, results: list):
    url = API_BASE + endpoint
    t0  = time.perf_counter()
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=8)) as r:
            await r.read()
            results.append(Result(endpoint, r.status, (time.perf_counter()-t0)*1000, time.time()))
    except Exception:
        results.append(Result(endpoint, 0, (time.perf_counter()-t0)*1000, time.time()))

async def ramp(
    endpoints   = ENDPOINTS,
    max_rps     = 40,
    duration_s  = 20,
    ramp_s      = 5,
):
    results = []
    start   = time.time()
    conn    = aiohttp.TCPConnector(limit=100)

    async with aiohttp.ClientSession(connector=conn) as session:
        while time.time() - start < duration_s:
            elapsed    = time.time() - start
            ramp_factor = min(elapsed / ramp_s, 1.0)
            current_rps = max(1, int(max_rps * ramp_factor))

            batch = [hit(session, ep, results)
                     for _ in range(current_rps)
                     for ep in endpoints]
            await asyncio.gather(*batch)
            await asyncio.sleep(1.0)

    return results

print('Running load test (20s ramp to 40 RPS)...')
results = await ramp()
print(f'  Total requests : {len(results)}')
print(f'  Success rate   : {sum(1 for r in results if 200<=r.status<300)/len(results)*100:.1f}%')
print(f'  Rate-limited   : {sum(1 for r in results if r.status==429)}')
print(f'  Errors         : {sum(1 for r in results if r.status==0 or r.status>=500)}')

In [ ]:
# ── Plot results ─────────────────────────────────────────────────────────────
df = pd.DataFrame([{'endpoint': r.endpoint, 'status': r.status,
                     'latency': r.latency, 'ts': r.ts} for r in results])
df['ok'] = df['status'].between(200, 299)
df['t']  = df['ts'] - df['ts'].min()

fig = plt.figure(figsize=(14, 8))
fig.suptitle('AxiraNews API — Load Test Results', color='#C8A97E', fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORS = {'#C8A97E': 0, '#5AC8FA': 1, '#28C840': 2}
EP_COLORS = dict(zip(ENDPOINTS, ['#C8A97E', '#5AC8FA', '#28C840']))

# Latency over time
ax = fig.add_subplot(gs[0, :])
for ep in ENDPOINTS:
    sub = df[df['endpoint']==ep]
    ax.scatter(sub['t'], sub['latency'], s=6, alpha=0.6, color=EP_COLORS[ep], label=ep)
    # Rolling P95
    if len(sub) > 10:
        roll = sub.set_index('t')['latency'].rolling('2s').quantile(0.95)
        ax.plot(roll.index, roll.values, color=EP_COLORS[ep], linewidth=1.5, alpha=0.8)
ax.set_title('Latency Over Time (dots=raw, line=P95 rolling)', color='#F0EDE6')
ax.set_xlabel('Seconds'); ax.set_ylabel('ms')
ax.legend(fontsize=8, framealpha=0.3); ax.grid(True, alpha=0.2)

# Latency distribution
ax2 = fig.add_subplot(gs[1, 0])
ok = df[df['ok']]['latency']
ax2.hist(ok, bins=30, color='#C8A97E', alpha=0.8, edgecolor='none')
for p, ls in [(50,'--'),(95,'-.'),(99,':')]:
    v = np.percentile(ok, p)
    ax2.axvline(v, color='#F0EDE6', linestyle=ls, linewidth=1, label=f'P{p}: {v:.0f}ms')
ax2.set_title('Latency Distribution', color='#F0EDE6')
ax2.set_xlabel('ms'); ax2.legend(fontsize=7, framealpha=0.3); ax2.grid(True, alpha=0.2, axis='y')

# Status code breakdown
ax3 = fig.add_subplot(gs[1, 1])
vc  = df['status'].value_counts()
c_map = {200:' #28C840', 429:'#FEBC2E', 0:'#FF5F57', 500:'#FF5F57'}
colors = [c_map.get(s, '#4A4A56') for s in vc.index]
ax3.bar([str(s) for s in vc.index], vc.values, color=colors, alpha=0.9)
ax3.set_title('Status Code Distribution', color='#F0EDE6')
ax3.set_xlabel('HTTP Status'); ax3.set_ylabel('Count'); ax3.grid(True, alpha=0.2, axis='y')

# RPS over time
ax4 = fig.add_subplot(gs[1, 2])
rps = df.groupby(df['t'].astype(int))['status'].count()
ax4.fill_between(rps.index, rps.values, alpha=0.3, color='#5AC8FA')
ax4.plot(rps.index, rps.values, color='#5AC8FA', linewidth=2)
ax4.set_title('Requests Per Second', color='#F0EDE6')
ax4.set_xlabel('Seconds'); ax4.set_ylabel('RPS'); ax4.grid(True, alpha=0.2)

plt.savefig('load_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print('\n── Summary ──────────────────────────────')
for ep in ENDPOINTS:
    sub = df[(df['endpoint']==ep) & df['ok']]['latency']
    if len(sub): print(f'{ep:<30} p50={np.percentile(sub,50):.0f}ms  p95={np.percentile(sub,95):.0f}ms  p99={np.percentile(sub,99):.0f}ms')